# W3 Day7: P1总复盘 + 工程深挖 (3)

> **日期**: 04/27 周日 | **时长**: 2.5h | **主题**: 白板复盘 + GNN图纸项目工程化

## 学习目标

| # | 目标 | 对应技能 |
|---|------|----------|
| 1 | 白板复盘 P1 全部核心概念 | Autograd / Transformer / CLIP / Diffusion / GAT |
| 2 | 掌握 GNN 工程决策: 建图、选型、标注 | CAD→Graph, GAT vs GCN, active learning |
| 3 | 设计 LLM 审查管线 | GNN → structured output → LLM code review |
| 4 | 撰写面试 STAR 脚本 | 用工程语言包装 GNN 项目 |

---

## 目录

1. **P1 知识地图** — W1~W3 全景回顾
2. **白板复盘: Autograd** — 计算图 / 链式法则 / forward/backward mode
3. **白板复盘: Transformer** — MHA / FFN / LayerNorm / residual stream
4. **白板复盘: CLIP** — 对比学习 / InfoNCE / zero-shot
5. **白板复盘: Diffusion** — 前向/逆向 / DDPM vs DDIM / CFG
6. **白板复盘: GAT** — 图注意力 / multi-head / vs GCN
7. **自测 Quiz** — 10 题速查
8. **工程深挖: 建图决策** — CAD→Graph 特征工程
9. **GAT vs GCN 选型** — 可解释性 vs 效率
10. **标注成本** — 半监督 / active learning / label propagation
11. **LLM 审查方案** — 端到端 pipeline
12. **扩展性** — 多楼层 / 模型服务 / 监控
13. **面试 STAR 脚本**

## 1. P1 知识地图

以下文字脑图回顾 W1~W3 所有核心知识点:

```
                        ┌──────────────────────────────────────────┐
                        │         P1: 深度学习基础 (W1~W3)         │
                        └──────────┬───────────┬───────────┬───────┘
                                   │           │           │
               ┌───────────────────┘           │           └───────────────────┐
               ▼                               ▼                               ▼
    ┌─────────────────────┐        ┌─────────────────────┐        ┌─────────────────────┐
    │   W1: Python/PyTorch│        │ W2: Transformer/    │        │   W3: GNN + 工程化   │
    │                     │        │     CLIP/Diffusion  │        │                     │
    │  - Python 并发      │        │  - Self-Attention   │        │  - GCN / GAT 原理    │
    │  - 元编程/设计模式   │        │  - Multi-Head Attn  │        │  - CAD→Graph 建图    │
    │  - Tensor 操作      │        │  - CLIP 对比学习    │        │  - Active Learning   │
    │  - Autograd 自动微分 │        │  - DDPM/DDIM 扩散   │        │  - LLM 审查管线      │
    │  - nn.Module/优化器 │        │  - CFG 引导         │        │  - 模型服务/监控      │
    │  - DataLoader/AMP   │        │  - Prompt Engineering│        │  - 面试 STAR 脚本    │
    └─────────────────────┘        └─────────────────────┘        └─────────────────────┘
```

**关键词**: `torch.autograd` · `nn.MultiheadAttention` · `InfoNCE` · `ε_θ(x_t, t)` · `GATConv` · `active learning` · `LLM code review`

## 2. 白板复盘: Autograd

### 2.1 计算图 (Computation Graph)

PyTorch 使用 **动态计算图** (define-by-run): 每次前向传播实时构建 DAG，反向传播后销毁。

$$\text{loss} = f(g(x, y), h(z))$$

```
x ──┬── g(x,y) ──┬── f(·) ── loss
y ──┘             │
z ─────────────── h(z) ──┘
```

### 2.2 链式法则 (Chain Rule)

反向传播的本质:

$$\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial f} \cdot \frac{\partial f}{\partial g} \cdot \frac{\partial g}{\partial x}$$

每条边存储 **局部梯度** $\frac{\partial \text{out}}{\partial \text{in}}$，从 loss 向叶子节点方向逐节点连乘。

### 2.3 Forward Mode vs Backward Mode

| 模式 | 方向 | 复杂度 | 适用场景 |
|------|------|--------|----------|
| Forward mode | 输入→输出 | $O(n)$ per input | 输入维数少，输出维数多 (如 Jacobian 列) |
| **Backward mode** | 输出→输入 | $O(1)$ per output | **深度学习**: 输出(loss)维=1，输入(params)维巨大 |

深度学习用 backward mode 的原因: loss 是标量，一次反传得到**所有参数**的梯度。

### 2.4 `torch.autograd.Function`

自定义可微操作需要实现 `forward()` 和 `backward()`:

```python
class MyReLU(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):              # ctx 保存中间结果
        ctx.save_for_backward(x)
        return x.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):    # grad_output = dL/dy
        x, = ctx.saved_tensors
        return grad_output * (x > 0).float()  # dL/dx = dL/dy * (x>0)
```

**面试关键点**: `requires_grad=True` → 叶子节点 → `.backward()` 触发拓扑排序 → 累加梯度到 `.grad`

## 3. 白板复盘: Transformer

### 3.1 核心公式

**Scaled Dot-Product Attention**:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Multi-Head Attention**:

$$\text{MHA}(Q,K,V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

### 3.2 Feed-Forward Network

$$\text{FFN}(x) = \text{ReLU}(x W_1 + b_1) W_2 + b_2$$

- 维度变化: $d_{\text{model}} \to d_{\text{ff}} \to d_{\text{model}}$，通常 $d_{\text{ff}} = 4 \times d_{\text{model}}$
- 近年常用 GELU / SwiGLU 替代 ReLU

### 3.3 LayerNorm & Residual Stream

$$\text{LN}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

**Pre-LN** (现代做法，训练更稳定):

$$x_{l+1} = x_l + \text{SubLayer}(\text{LN}(x_l))$$

Residual stream 视角: 每层在**同一通道**上叠加信息，$d_{\text{model}}$ 是信息带宽。

### 3.4 Scaling Law

$$\text{Performance} \propto N^{\alpha_N} \cdot D^{\alpha_D} \cdot C^{\alpha_C}$$

- $N$: 模型参数量, $D$: 数据量, $C$: 计算量
- $\sqrt{d_k}$ 缩放防止 softmax 饱和
- 训练时常用 **warmup + cosine decay** 学习率调度

## 4. 白板复盘: CLIP

### 4.1 对比学习 (Contrastive Learning)

CLIP 同时训练 **图像编码器** $f_\theta$ 和 **文本编码器** $g_\phi$，使匹配对的嵌入靠近，不匹配对的嵌入远离。

### 4.2 InfoNCE Loss

给定 batch 中 $N$ 个图文对 $\{(I_i, T_i)\}$:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \left[ \log \frac{\exp(\text{sim}(I_i, T_i)/\tau)}{\sum_{j=1}^{N} \exp(\text{sim}(I_i, T_j)/\tau)} + \log \frac{\exp(\text{sim}(T_i, I_i)/\tau)}{\sum_{j=1}^{N} \exp(\text{sim}(T_j, I_i)/\tau)} \right]$$

其中 $\text{sim}(a,b) = \frac{a^T b}{\|a\| \|b\|}$ 是余弦相似度，$\tau$ 是可学习温度参数。

### 4.3 Zero-Shot 分类

```
1. 构建 prompt: ["a photo of a cat", "a photo of a dog", ...]
2. 文本编码器 → text embeddings (N_classes × d)
3. 图像编码器 → image embedding (1 × d)
4. 余弦相似度 → softmax → 预测类别
```

### 4.4 Prompt Engineering

- **Prompt template**: `"a photo of a {class}."` 远优于裸标签
- **Ensemble of prompts**: 对每个类别用多个 prompt 取平均嵌入
- **可学习 prompt**: CoOp (Context Optimization) — 用可训练 token 替代模板

## 5. 白板复盘: Diffusion

### 5.1 前向过程 (Forward / Diffusion Process)

逐步向数据添加高斯噪声:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t I)$$

**重参数化技巧** (一步到位):

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t}\, x_0,\; (1-\bar{\alpha}_t) I)$$

其中 $\alpha_t = 1 - \beta_t$，$\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$。

### 5.2 逆向过程 (Reverse / Denoising Process)

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t),\; \sigma_t^2 I)$$

网络 $\epsilon_\theta(x_t, t)$ 预测噪声 $\epsilon$，训练目标:

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon} \left[ \|\epsilon - \epsilon_\theta(x_t, t)\|^2 \right]$$

### 5.3 DDPM vs DDIM

| | DDPM | DDIM |
|---|------|------|
| 采样步数 | $T=1000$ | 可选 $S \ll T$ |
| 采样方式 | 随机 (stochastic) | 确定性 (deterministic) |
| 速度 | 慢 | **快 10~50 倍** |
| 可逆性 | 否 | **是** (可用于编辑) |

### 5.4 Classifier-Free Guidance (CFG)

同时训练有条件 $\epsilon_\theta(x_t, t, c)$ 和无条件 $\epsilon_\theta(x_t, t, \varnothing)$:

$$\tilde{\epsilon} = \epsilon_\theta(x_t, t, \varnothing) + w \cdot [\epsilon_\theta(x_t, t, c) - \epsilon_\theta(x_t, t, \varnothing)]$$

- $w > 1$ 时增强条件引导，生成更符合文本描述的图像
- $w=7.5$ 是 Stable Diffusion 常用值

## 6. 白板复盘: GAT (Graph Attention Network)

### 6.1 图注意力机制

对于节点 $i$ 和邻居 $j \in \mathcal{N}(i)$:

$$e_{ij} = \text{LeakyReLU}(\mathbf{a}^T [W h_i \| W h_j])$$

$$\alpha_{ij} = \text{softmax}(e_{ij}) = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}$$

$$h_i' = \sigma\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij} W h_j\right)$$

### 6.2 Multi-Head Attention

$$h_i' = \|_{k=1}^{K} \sigma\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij}^k W^k h_j\right)$$

- 输出层拼接 $K$ 个 head 的结果
- 最后一层可改为平均 (类似 Transformer 的 MHA)

### 6.3 GAT vs GCN

| | GCN | GAT |
|---|-----|-----|
| 聚合权重 | 固定: $\frac{1}{\sqrt{d_i d_j}}$ | **学习**: attention $\alpha_{ij}$ |
| 可解释性 | 低 | **高**: attention 权重可视化 |
| 参数量 | 少 | 多 (每个 head 有 $\mathbf{a}$ 向量) |
| 适合场景 | 同质图 (邻居重要性相近) | **异质图** (邻居重要性差异大) |
| 工程审核 | 一般 | **强**: 能看到模型关注哪些连接 |

**对于工程图纸审核项目**: GAT 的 attention 权重可以直接告诉我们模型关注了哪些构件连接，这对安全审查非常重要。

In [ ]:
# ============================================================
# Cell 7: P1 自测 Quiz — 10 道核心概念题
# ============================================================

quiz_questions = [
    {
        'q': 'Autograd 中 backward mode 相比 forward mode 的核心优势是什么？',
        'a': '一次反向传播即可得到所有参数的梯度 (O(1) per output)，\n'
              '      因为 loss 是标量。Forward mode 需要对每个输入分别计算。'
    },
    {
        'q': 'Transformer 中为什么用 \u221ad_k 缩放点积？',
        'a': '当 d_k 较大时，点积值的方差为 d_k，导致 softmax 进入饱和区。\n'
              '      除以 \u221ad_k 使方差回到 1，梯度更稳定。'
    },
    {
        'q': 'CLIP 的 InfoNCE loss 中温度参数 \u03c4 的作用？',
        'a': '\u03c4 控制 softmax 分布的锐度。\u03c4 小\u2192分布更尖锐\u2192对困难负样本更严格；\n'
              '      \u03c4 大\u2192分布更平滑。\u03c4 是可学习参数。'
    },
    {
        'q': 'DDPM 训练目标 L_simple 为什么预测噪声 \u03b5 而不是直接预测 x_0？',
        'a': '预测噪声 \u03b5 的 loss 形式更简洁，且实验效果更好。\n'
              '      从 x_t 和预测的 \u03b5 可以算出 x_0 的估计。'
    },
    {
        'q': 'Classifier-Free Guidance 中 w > 1 的效果？',
        'a': 'w > 1 增强条件引导，放大有条件与无条件预测的差异。\n'
              '      生成结果更符合文本描述但多样性降低。'
    },
    {
        'q': 'GAT 中 attention 权重 \u03b1_ij 的计算包含哪些步骤？',
        'a': '(1) 线性变换 W*h_i, W*h_j (2) 拼接后与向量 a 做内积\n'
              '      (3) LeakyReLU 激活 (4) 对邻居做 softmax 归一化。'
    },
    {
        'q': 'Transformer 中 Pre-LN 和 Post-LN 的区别？',
        'a': 'Post-LN: x + SubLayer(LN(x)) 后再 LN \u2192 原始论文，需 warmup。\n'
              '      Pre-LN: x + SubLayer(LN(x)) \u2192 训练更稳定，现为主流。'
    },
    {
        'q': 'CLIP zero-shot 分类为什么需要 prompt template？',
        'a': '训练时 CLIP 看到的是句子而非孤立词汇。Prompt template (如 "a photo of a {c}")\n'
              '      使推理时的文本分布与训练时一致，显著提升准确率。'
    },
    {
        'q': 'DDIM 如何实现加速采样？',
        'a': 'DDIM 将采样过程变为确定性 ODE，可跳步采样 (如 1000\u219250 步)。\n'
              '      选择时间步子集 {\u03c4_1, ..., \u03c4_S} 进行去噪，每步仍用同一个模型。'
    },
    {
        'q': 'GNN 中图注意力相比 GCN 固定权重的优势？',
        'a': 'GAT 学习每条边的注意力权重，可处理邻居重要性不均匀的异质图。\n'
              '      注意力权重天然提供可解释性，适合安全审查场景。'
    },
]

# ---- 输出 Quiz ----
colors = {'header': '#c2553a', 'question': '#5a6b4a', 'answer': '#2d2a26'}

print(f"{'='*60}")
print(f"  P1 总复盘: 自测 Quiz (共 {len(quiz_questions)} 题)")
print(f"{'='*60}\n")

for i, item in enumerate(quiz_questions, 1):
    print(f"\033[1;31m  Q{i}. {item['q']}\033[0m")
    print(f"\033[90m  (先自己回答，再展开答案)\033[0m")
    print(f"  [答案] {item['a']}")
    print(f"  {'-'*56}")

print(f"\n{'='*60}")
print(f"  Quiz 完成! 回顾不熟悉的题目，标记重点复习。")
print(f"{'='*60}")

## 7. 工程深挖: 建图决策 (CAD → Graph)

### 7.1 将工程图纸转为图结构

```
CAD 图纸 (.dwg/.dxf)
        │
        ▼ 解析
┌──────────────────────────┐
│  识别构件 (门/窗/墙/柱)   │  → Node 特征
│  识别空间关系 (连接/相邻) │  → Edge 特征
│  提取属性 (尺寸/材质/编号)│  → Node 属性
└──────────────────────────┘
        │
        ▼
   异质图 G = (V, E, X, A)
```

### 7.2 Node 设计

| 节点类型 | 特征向量 | 维度示例 |
|----------|----------|----------|
| 墙 (Wall) | 长度、厚度、材质编码、朝向 | 32 |
| 门 (Door) | 宽度、高度、类型编码、防火等级 | 32 |
| 窗 (Window) | 宽度、高度、玻璃类型 | 32 |
| 柱 (Column) | 截面尺寸、承载力、材质 | 32 |

### 7.3 Edge 设计

| 边类型 | 含义 | 特征 |
|--------|------|------|
| 连接 (adjacent) | 构件物理接触 | 接触面积、连接方式 |
| 支撑 (support) | 结构承重关系 | 荷载传递方向 |
| 包含 (contain) | 门窗在墙内 | 相对位置 |
| 相邻 (neighbor) | 空间上靠近 | 距离、是否同层 |

### 7.4 特征工程注意事项

- **归一化**: 尺寸特征做 MinMax 或 Z-score
- **类别编码**: 材质/类型用 one-hot 或 learnable embedding
- **缺失值**: 用 0 填充 + 添加 mask 特征位
- **异质图**: 不同类型节点/边使用不同的 GNN 层

## 8. GAT vs GCN 选型

### 8.1 为什么选 GAT？

**工程审核场景的核心需求: 可解释性**

- 审核人员需要知道: 模型为什么标记这个区域有问题？
- GAT 的 attention 权重 $\alpha_{ij}$ 天然给出**每个邻居的贡献度**
- 可以高亮显示模型关注的连接 → 辅助人工审查

```
              Wall_1 ──α=0.45── Door_1 (高注意力: 防火门连接异常)
               │
              α=0.30
               │
              Column_2 ──α=0.25── Wall_3
```

### 8.2 什么时候 GCN 更好？

| 场景 | 推荐 | 原因 |
|------|------|------|
| 同质图，邻居重要性相近 | **GCN** | 参数少，计算快 |
| 大规模图 (>10万节点) | **GCN** | GAT 的 attention 计算量大 |
| 需要快速原型验证 | **GCN** | 实现简单，基线效果好 |
| 安全审查 / 需要可解释 | **GAT** | attention 可视化 |
| 异质图 (多种节点类型) | **GAT** | 自适应权重分配 |

### 8.3 实际工程方案

```
推荐: GAT (主模型) + GCN (快速预筛)

Stage 1: GCN 快速扫描全图 → 标记可疑区域
Stage 2: GAT 精细审查可疑区域 → 输出 attention 热力图
Stage 3: LLM 结合 GNN 结果生成审核报告
```

## 9. 标注成本与半监督学习

### 9.1 标注成本问题

工程图纸标注需要**专业工程师**，成本极高:

| 标注方式 | 单张成本 | 质量 |
|----------|----------|------|
| 初级标注员 | \u00a550-100 | 需审核 |
| 专业工程师 | \u00a5300-500 | 高 |
| 专家审核 | \u00a5500-1000 | 最高 |

### 9.2 三种降低标注成本的策略

**① 半监督学习 (Semi-Supervised)**

- 核心思想: 少量标注 + 大量未标注数据
- 方法: 伪标签 (pseudo-label)、一致性正则化、自训练
- GNN 特有: 利用图结构传播标签 (label propagation)

**② 主动学习 (Active Learning)**

- 核心思想: 让模型**选择最值得标注的样本**
- 策略: 不确定性采样 (uncertainty sampling)、多样性采样
- 目标: 用 10% 标注量达到 90% 标注量的效果

**③ 标签传播 (Label Propagation)**

$$Y^{(t+1)} = \alpha \cdot A Y^{(t)} + (1-\alpha) Y^{(0)}$$

- 利用图的邻接矩阵 $A$ 将已标注节点的标签传播到未标注节点
- 适合: 同质图 (相连节点倾向同标签)

In [ ]:
# ============================================================
# Cell 11: Active Learning 模拟实验
# 用合成数据演示: 选择最有价值的样本标注
# ============================================================

import numpy as np
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ---- 颜色定义 ----
C_RED    = '#c2553a'
C_GREEN  = '#5a6b4a'
C_DARK   = '#2d2a26'

# ---- 生成合成图数据 (模拟工程图纸节点) ----
n_samples = 200
n_labeled = 10     # 初始标注数

# 两个特征: [构件密度, 连接复杂度]
X = np.random.randn(n_samples, 2)
# 真实标签: 根据到原点的距离判断是否有问题
y_true = (X[:, 0]**2 + X[:, 1]**2 > 2.0).astype(int)

# 添加一些噪声使任务更真实
flip_mask = np.random.rand(n_samples) < 0.1
y_true[flip_mask] = 1 - y_true[flip_mask]

# ---- 简单分类器 (模拟 GNN 预测) ----
try:
    from scipy.spatial.distance import cdist

    class KNNClassifier:
        """简化分类器模拟 GNN 的不确定性估计"""
        def __init__(self, k=5):
            self.k = k
            self.X_train = None
            self.y_train = None
        
        def fit(self, X, y):
            self.X_train = X.copy()
            self.y_train = y.copy()
        
        def predict_proba(self, X):
            if self.X_train is None or len(self.X_train) == 0:
                return np.ones((len(X), 2)) * 0.5
            dists = cdist(X, self.X_train)
            probs = []
            for i in range(len(X)):
                idx = np.argsort(dists[i])[:self.k]
                labels = self.y_train[idx]
                p1 = np.mean(labels)
                probs.append([1 - p1, p1])
            return np.array(probs)
        
        def predict(self, X):
            return (self.predict_proba(X)[:, 1] > 0.5).astype(int)

    # ---- Active Learning 循环 ----
    def uncertainty_sampling(proba, n_select=5):
        """选择最不确定的样本 (概率接近 0.5)"""
        uncertainty = 1 - np.abs(proba[:, 1] - 0.5) * 2  # 0=确定, 1=最不确定
        return np.argsort(uncertainty)[-n_select:]

    def random_sampling(n_pool, n_select=5):
        """随机选择 (baseline)"""
        return np.random.choice(n_pool, n_select, replace=False)

    # 初始标注集
    initial_idx = np.random.choice(n_samples, n_labeled, replace=False)

    # Active Learning
    labeled_set_al = set(initial_idx.tolist())
    pool_set_al = list(set(range(n_samples)) - labeled_set_al)

    # Random baseline
    labeled_set_rd = set(initial_idx.tolist())
    pool_set_rd = list(set(range(n_samples)) - labeled_set_rd)

    rounds = 15
    n_per_round = 5
    acc_active = []
    acc_random = []
    n_labeled_list = []

    for r in range(rounds):
        # ---- Active Learning ----
        clf_al = KNNClassifier(k=5)
        labeled_arr = np.array(list(labeled_set_al))
        clf_al.fit(X[labeled_arr], y_true[labeled_arr])
        proba_al = clf_al.predict_proba(X)
        y_pred_al = clf_al.predict(X)
        acc_al = np.mean(y_pred_al == y_true)
        acc_active.append(acc_al)
        
        # 选择最不确定的样本
        pool_arr = np.array(pool_set_al)
        if len(pool_arr) >= n_per_round:
            uncertain = pool_arr[uncertainty_sampling(proba_al[pool_arr], n_per_round)]
            for idx in uncertain:
                labeled_set_al.add(int(idx))
            pool_set_al = list(set(range(n_samples)) - labeled_set_al)
        
        # ---- Random ----
        clf_rd = KNNClassifier(k=5)
        labeled_arr_rd = np.array(list(labeled_set_rd))
        clf_rd.fit(X[labeled_arr_rd], y_true[labeled_arr_rd])
        y_pred_rd = clf_rd.predict(X)
        acc_rd = np.mean(y_pred_rd == y_true)
        acc_random.append(acc_rd)
        
        if len(pool_set_rd) >= n_per_round:
            rand_idx = random_sampling(len(pool_set_rd), n_per_round)
            selected = [pool_set_rd[i] for i in rand_idx]
            for idx in selected:
                labeled_set_rd.add(idx)
            pool_set_rd = list(set(range(n_samples)) - labeled_set_rd)
        
        n_labeled_list.append(len(labeled_set_al))

    # ---- 纯文本输出 ----
    print(f"{'='*60}")
    print(f"  Active Learning 模拟实验")
    print(f"{'='*60}")
    print(f"  总样本数: {n_samples} | 初始标注: {n_labeled}")
    print(f"  每轮新增标注: {n_per_round}")
    print()

    print(f"  {'已标注':>6} | {'Active Learning':>16} | {'Random':>16} | {'提升':>8}")
    print(f"  {'-'*6} | {'-'*16} | {'-'*16} | {'-'*8}")

    for i in range(len(acc_active)):
        diff = acc_active[i] - acc_random[i]
        marker = ' *' if diff > 0.02 else ''
        print(f"  {n_labeled_list[i]:>6} | {acc_active[i]:>14.1%}   | {acc_random[i]:>14.1%}   | {diff:>+7.1%}{marker}")

    print()
    print(f"  结论: Active Learning 用更少的标注达到更高准确率")
    print(f"  最终 Active Learning 准确率: {acc_active[-1]:.1%}")
    print(f"  最终 Random 准确率:          {acc_random[-1]:.1%}")
    print(f"{'='*60}")

except ImportError:
    print("  [!] scipy 未安装，用纯 numpy 替代演示")
    # 简化版不依赖 scipy
    n_rounds = 15
    print(f"  总样本: {n_samples}, 初始标注: {n_labeled}")
    print(f"  模拟 {n_rounds} 轮主动学习...")
    for r in range(n_rounds):
        current_labeled = n_labeled + r * 5
        # 理论上 active learning 收敛更快
        al_acc = min(0.95, 0.55 + r * 0.03 + 0.05)
        rd_acc = min(0.90, 0.55 + r * 0.025)
        diff = al_acc - rd_acc
        marker = ' *' if diff > 0.02 else ''
        print(f"  {current_labeled:>6} | {al_acc:>14.1%}   | {rd_acc:>14.1%}   | {diff:>+7.1%}{marker}")
    print(f"  结论: Active Learning 可节省约 50-70% 的标注成本")
    print(f"{'='*60}")

## 10. LLM 审查方案: 端到端 Pipeline

### 10.1 完整管线设计

```
┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│  CAD 图纸 │───→│ 建图模块  │───→│  GNN 模型 │───→│ 结构化   │───→│  LLM     │
│  (.dwg)  │    │ CAD→Graph│    │ GAT/GCN  │    │  输出    │    │ 审查报告 │
└──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
                                                                     │
                                                                     ▼
                                                              ┌──────────┐
                                                              │ 人工复核  │
                                                              │ (可选)   │
                                                              └──────────┘
```

### 10.2 各阶段详细设计

| 阶段 | 输入 | 输出 | 技术方案 |
|------|------|------|----------|
| 建图 | CAD 文件 | NetworkX/DGL Graph | 规则引擎 + CV 检测 |
| GNN 推理 | 图结构 + 节点特征 | 节点/边异常分数 | GAT (可解释) |
| 结构化 | GNN 输出 | JSON 报告 | 阈值过滤 + attention 可视化 |
| LLM 审查 | JSON + 图纸描述 | 审查意见书 | GPT-4 / Claude + RAG |

### 10.3 GNN → LLM 的结构化输出格式

```json
{
  "drawing_id": "ARCH-2024-0427-F3",
  "total_nodes": 156,
  "anomalies": [
    {
      "node_id": "wall_23",
      "type": "fire_rating_mismatch",
      "confidence": 0.87,
      "attention_neighbors": ["door_15", "column_8"],
      "description": "\u5899\u4f53\u9632\u706b\u7b49\u7ea7\u4e0e\u76f8\u90bb\u9632\u706b\u95e8\u4e0d\u5339\u914d"
    }
  ],
  "summary": {"high_risk": 2, "medium_risk": 5, "low_risk": 12}
}
```

In [ ]:
# ============================================================
# Cell 13: LLM 审查 Prompt 模板生成器
# 将 GNN 结构化输出组装为 LLM 可用的 prompt
# ============================================================

import json
from datetime import datetime

# ---- 颜色 ----
C_RED    = '\033[91m'
C_GREEN  = '\033[92m'
C_DARK   = '\033[90m'
C_RESET  = '\033[0m'
C_BOLD   = '\033[1m'

# ---- 模拟 GNN 输出 (结构化结果) ----
gnn_output = {
    "drawing_id": "ARCH-2024-0427-F3",
    "building": "\u67d0\u5546\u4e1a\u7efc\u5408\u4f53",
    "floor": 3,
    "total_nodes": 156,
    "total_edges": 412,
    "anomalies": [
        {
            "node_id": "wall_23",
            "node_type": "wall",
            "anomaly_type": "fire_rating_mismatch",
            "confidence": 0.87,
            "attention_neighbors": ["door_15 (alpha=0.45)", "column_8 (alpha=0.30)"],
            "feature_highlights": {
                "wall_fire_rating": "B1",
                "adjacent_door_fire_rating": "A",
                "code_requirement": "GB 50016-2014: \u9632\u706b\u5899\u5e94\u4e3a A \u7ea7"
            }
        },
        {
            "node_id": "door_31",
            "node_type": "door",
            "anomaly_type": "exit_width_insufficient",
            "confidence": 0.72,
            "attention_neighbors": ["corridor_12 (alpha=0.52)", "wall_28 (alpha=0.28)"],
            "feature_highlights": {
                "door_width": 0.9,
                "code_requirement": "GB 50016-2014: \u758f\u6563\u95e8\u51c0\u5bbd\u4e0d\u5e94\u5c0f\u4e8e 0.9m",
                "occupant_count": 120
            }
        },
        {
            "node_id": "column_8",
            "node_type": "column",
            "anomaly_type": "load_path_discontinuity",
            "confidence": 0.65,
            "attention_neighbors": ["beam_4 (alpha=0.40)", "wall_23 (alpha=0.35)"],
            "feature_highlights": {
                "column_section": "400x400mm",
                "support_below": "none_detected",
                "code_requirement": "GB 50010-2010: \u67f1\u5e94\u6709\u8fde\u7eed\u4f20\u529b\u8def\u5f84"
            }
        }
    ],
    "summary": {"high_risk": 1, "medium_risk": 1, "low_risk": 1},
    "model_info": {
        "gnn_type": "GAT",
        "heads": 4,
        "layers": 3,
        "overall_confidence": 0.75
    }
}


# ---- Prompt 模板生成器 ----
def build_llm_review_prompt(gnn_output: dict, role: str = '\u7ed3\u6784\u5de5\u7a0b\u5e08') -> str:
    """
    将 GNN 结构化输出组装为 LLM 审查 prompt。
    
    Args:
        gnn_output: GNN 模型输出的 JSON 结构
        role: 审查角色 (结构工程师 / 建筑师 / 消防审查员)
    
    Returns:
        完整的 LLM prompt 字符串
    """
    # 系统指令
    system_prompt = f"""\u4f60\u662f\u4e00\u4f4d\u8d44\u6df1{role}\uff0c\u8d1f\u8d23\u5ba1\u67e5\u5efa\u7b51\u56fe\u7eb8\u3002
\u4f60\u5c06\u6536\u5230\u4e00\u4e2a\u7531 AI \u56fe\u795e\u7ecf\u7f51\u7edc (GNN) \u81ea\u52a8\u68c0\u6d4b\u7684\u7ed3\u6784\u5316\u62a5\u544a\u3002
\u8bf7\u6839\u636e\u4e2d\u56fd\u5efa\u7b51\u89c4\u8303\u5bf9\u6bcf\u4e2a\u5f02\u5e38\u9879\u8fdb\u884c\u4e13\u4e1a\u5224\u65ad\u3002

\u8f93\u51fa\u8981\u6c42:
1. \u5bf9\u6bcf\u4e2a\u5f02\u5e38\u7ed9\u51fa: \u786e\u8ba4/\u5426\u5b9a/\u5f85\u5b9a \u5224\u5b9a
2. \u5f15\u7528\u5177\u4f53\u89c4\u8303\u6761\u6587
3. \u7ed9\u51fa\u6574\u6539\u5efa\u8bae
4. \u6309\u98ce\u9669\u7b49\u7ea7\u6392\u5e8f"""
    
    # 图纸信息
    drawing_info = f"""
--- 图纸信息 ---
图纸编号: {gnn_output['drawing_id']}
建筑名称: {gnn_output['building']}
楼层: {gnn_output['floor']}F
构件总数: {gnn_output['total_nodes']} 个节点, {gnn_output['total_edges']} 条边
AI 检测置信度: {gnn_output['model_info']['overall_confidence']:.0%}
GNN 模型: {gnn_output['model_info']['gnn_type']} ({gnn_output['model_info']['heads']} heads, {gnn_output['model_info']['layers']} layers)"""
    
    # 异常详情
    anomaly_sections = []
    for i, anom in enumerate(gnn_output['anomalies'], 1):
        section = f"""
--- 异常 #{i} ---
构件: {anom['node_id']} ({anom['node_type']})
问题类型: {anom['anomaly_type']}
AI 置信度: {anom['confidence']:.0%}
注意力关联构件: {', '.join(anom['attention_neighbors'])}
关键参数:"""
        for k, v in anom['feature_highlights'].items():
            section += f"\n  - {k}: {v}"
        anomaly_sections.append(section)
    
    # 汇总
    summary = f"""
--- 检测汇总 ---
高风险: {gnn_output['summary']['high_risk']} 项
中风险: {gnn_output['summary']['medium_risk']} 项
低风险: {gnn_output['summary']['low_risk']} 项

请对以上异常逐项进行专业审查，并输出结构化审查意见。"""
    
    return system_prompt + drawing_info + ''.join(anomaly_sections) + summary


# ---- 生成并输出 ----
try:
    prompt = build_llm_review_prompt(gnn_output, role='\u7ed3\u6784\u5de5\u7a0b\u5e08')
    
    print(f"{C_BOLD}{C_RED}{'='*65}{C_RESET}")
    print(f"{C_BOLD}  LLM 审查 Prompt 模板生成器{C_RESET}")
    print(f"{C_RED}{'='*65}{C_RESET}\n")
    
    # 统计信息
    print(f"{C_GREEN}  [统计]{C_RESET}")
    print(f"  - Prompt 总长度: {len(prompt)} 字符")
    print(f"  - 异常数量: {len(gnn_output['anomalies'])} 项")
    print(f"  - 涉及规范: GB 50016-2014, GB 50010-2010")
    print(f"  - 估计 token 数: ~{len(prompt) // 2} tokens\n")
    
    print(f"{C_BOLD}{C_DARK}{chr(9472)*65}{C_RESET}")
    print(f"{C_BOLD}  生成的 Prompt:{C_RESET}\n")
    print(prompt)
    print(f"\n{C_BOLD}{C_DARK}{chr(9472)*65}{C_RESET}")
    
    # 同时组装 JSON 方便后续使用
    output_package = {
        'system_role': '\u4f60\u662f\u8d44\u6df1\u7ed3\u6784\u5de5\u7a0b\u5e08\uff0c\u8d1f\u8d23\u5ba1\u67e5\u5efa\u7b51\u56fe\u7eb8\u3002',
        'user_prompt': prompt,
        'metadata': {
            'generated_at': datetime.now().isoformat(),
            'gnn_output': gnn_output,
            'prompt_length': len(prompt),
            'estimated_tokens': len(prompt) // 2
        }
    }
    
    print(f"\n{C_GREEN}  [OK] Prompt 包已组装完成，可直接发送给 LLM API{C_RESET}")
    print(f"{C_GREEN}  [OK] 结构: {{system_role, user_prompt, metadata}}{C_RESET}")
    
except Exception as e:
    print(f"{C_RED}  [X] 生成失败: {e}{C_RESET}")

# ---- 扩展: 批量生成不同角色的 prompt ----
print(f"\n{C_BOLD}  [多角色 Prompt 对比]{C_RESET}")
roles = ['\u7ed3\u6784\u5de5\u7a0b\u5e08', '\u5efa\u7b51\u8bbe\u8ba1\u5e08', '\u6d88\u9632\u5ba1\u67e5\u5458']
for role in roles:
    p = build_llm_review_prompt(gnn_output, role=role)
    print(f"  - {role}: {len(p)} 字符, ~{len(p)//2} tokens")

print(f"\n{C_RED}{'='*65}{C_RESET}")

## 11. 扩展性设计

### 11.1 多楼层 / 多建筑支持

```
单层图纸 GNN          →          多层建筑 GNN

     G₁ (1F)                          Building Graph
                                       ├─ Floor_1
     G₂ (2F)           →             │   └─ G₁
                                       ├─ Floor_2
     G₃ (3F)                          │   └─ G₂
                                       ├─ Floor_3
  独立推理，各自报告                    │   └─ G₃
                                       └─ Vertical Edges (楼梯/电梯)
                                      全楼统一推理
```

### 11.2 模型服务架构

| 组件 | 技术方案 | 作用 |
|------|----------|------|
| 模型服务 | TorchServe / Triton | GNN 模型部署，GPU 推理 |
| 任务队列 | Redis + Celery | 异步处理大图纸 |
| 存储 | PostgreSQL + Neo4j | 结构化数据 + 图数据 |
| API 网关 | FastAPI | RESTful 接口 |
| 前端 | React + D3.js | 图纸可视化 + attention 热力图 |

### 11.3 监控与持续改进

- **数据漂移检测**: 监控不同设计院图纸的分布变化
- **反馈闭环**: 工程师纠正 → 自动加入训练集 → 定期微调
- **A/B 测试**: GAT v1 vs GAT v2 并行推理，比较准确率
- **延迟监控**: P99 推理延迟 < 5s (单层图纸)

## 12. 面试 STAR 脚本: GNN 图纸审核项目

### STAR 框架

| 环节 | 内容 |
|------|------|
| **S**ituation | 建筑工程审核依赖人工，效率低且易遗漏。某项目有 500+ 张图纸需要审查。 |
| **T**ask | 设计 AI 辅助审核系统，自动检测图纸中的规范违规问题。 |
| **A**ction | 1) 设计 CAD→Graph 建图方案 (节点=构件，边=空间关系)<br>2) 选用 GAT 模型 (可解释性强，attention 可定位关键连接)<br>3) 主动学习策略降低 70% 标注成本<br>4) 设计 GNN→LLM 管线生成专业审查报告 |
| **R**esult | 检测准确率 85%+，审核效率提升 5 倍，attention 可视化获得工程师好评。 |

### 面试话术模板

```markdown
面试官: 介绍一下你做过的 GNN 项目。

回答:
"我在项目中做了一个建筑工程图纸的 AI 审核系统。背景是建筑行业的图纸审核
目前主要靠人工，效率低且容易遗漏，尤其是一些隐蔽的规范冲突。

我的方案是把 CAD 图纸转成图结构——节点是建筑构件（墙、门、窗、柱），
边是它们之间的空间关系（连接、支撑、包含）。然后用 GAT 做异常检测。

选 GAT 而不是 GCN 的核心原因是可解释性。工程审核场景下，你不仅要
发现问题，还要告诉工程师为什么有问题。GAT 的 attention 权重可以直接
高亮显示模型关注了哪些构件连接，工程师一看就懂。

标注方面，专业工程师标注成本很高，所以我用了主动学习策略——让模型
选择最不确定的样本优先标注，最终用 30% 的标注量达到了 90% 标注量
的效果。

最后我把 GNN 的结构化输出接入 LLM，生成符合中国建筑规范的审查意见书，
形成端到端的自动化管线。"
```

### 可能的追问及回答

| 追问 | 回答要点 |
|------|----------|
| 图是怎么建的？ | 节点=构件 (规则解析CAD实体)，边=空间关系 (距离+拓扑) |
| 为什么不用 CV？ | 图结构天然编码拓扑关系，CV 只能看像素，不理解连接 |
| 数据量多大？ | 初期 200 张图纸，约 3 万节点，用主动学习扩展 |
| GNN 层数？ | 3 层 GAT，4 heads，避免 over-smoothing |
| 如何评估？ | Precision/Recall/F1，重点关注 false negative (漏检) |

In [ ]:
# ============================================================
# Cell 16: P1 阶段完成总结
# ============================================================

print("\033[1;31m" + "=" * 60 + "\033[0m")
print("\033[1;31m  P1 阶段完成! 深度学习基础 — 全部掌握\033[0m")
print("\033[1;31m" + "=" * 60 + "\033[0m")

achievements = [
    ("W1: Python 工程", "并发编程 / 元编程 / 设计模式", "#c2553a"),
    ("W1: PyTorch 基础", "Tensor / Autograd / Module / DataLoader", "#c2553a"),
    ("W2: Transformer", "Self-Attention / MHA / FFN / LayerNorm", "#5a6b4a"),
    ("W2: CLIP", "对比学习 / InfoNCE / Zero-Shot / Prompt", "#5a6b4a"),
    ("W2: Diffusion", "DDPM / DDIM / CFG / epsilon_theta 预测", "#5a6b4a"),
    ("W3: GNN", "GCN / GAT / 图注意力 / 多头机制", "#2d2a26"),
    ("W3: 工程化", "CAD->Graph / Active Learning / LLM 审查", "#2d2a26"),
    ("W3: 面试准备", "STAR 脚本 / 项目话术 / 追问应对", "#2d2a26"),
]

print()
for title, details, color in achievements:
    print(f"  \033[1m[+] {title}\033[0m")
    print(f"      {details}")

print()
print("  " + "-" * 56)
print()

stats = {
    '总学习天数': 7,
    '笔记本数量': 7,
    '核心概念': 30,
    '代码示例': 20,
    '面试脚本': '3 (Transformer/CLIP/GNN)',
}

print("  P1 阶段数据统计:")
for k, v in stats.items():
    print(f"    {k}: {v}")

print()
print("  " + "=" * 56)
print("  下一步: P2 阶段 — 进阶专题 + 项目实战")
print("  " + "=" * 56)

# ---- 概念掌握度自评 ----
print("\n\033[1;31m  [概念掌握度自评]\033[0m")
mastery = {
    'Autograd (计算图/链式法则)':  '########## 90%%',
    'Transformer (MHA/FFN/LN)':     '#########_ 85%%',
    'CLIP (对比学习/Zero-Shot)':     '########__ 80%%',
    'Diffusion (DDPM/CFG)':         '#######___ 70%%',
    'GAT (图注意力/多头)':           '########__ 80%%',
    '工程化 (建图/Active Learning)':  '#######___ 70%%',
}

for concept, bar in mastery.items():
    print(f"    {concept:35s} {bar}")

print("\n\033[1;31m" + "=" * 60 + "\033[0m")
print("\033[1;31m  P1 Complete! Ready for P2!\033[0m")
print("\033[1;31m" + "=" * 60 + "\033[0m")